Assume control outcomes at time $t\in 1,\ldots, T$ for unit $i\in1,\ldots, N$ are generated according to:
$$
\vec{y}_{it} = \mathbf{\Lambda}_i^\top \vec{f_t} + \vec{\varepsilon}_{it}.
$$ 
Let the matrix $\mathbf{\Lambda}_i = [\vec{\lambda}_{i1},\ldots,\vec{\lambda}_{iM}]$, where the $\lambda$ are column vectors, and $M$ is the number of outcomes. Note: $\vec{y}_{it}\in \mathbb{R}^{M\times 1}$, $\mathbf{\Lambda}_i \in \mathbb{R}^{k\times M}$, $\vec{f}_t\in\mathbb{R}^{k\times 1}$, and $\vec{\varepsilon}_{it}\in \mathbb{R}^{M\times 1}$, where $k$ is the number of factors. Define the matrix of outcomes at time $t$ as:
$$
\mathbf{Y}_t = (\vec{y}_{1t},\ldots, \vec{y}_{Nt})^\top = (\vec{f}_t^\top \mathbf{\Lambda}_1,\ldots, \vec{f}_t^\top \mathbf{\Lambda}_N)^\top= \begin{pmatrix}
    y_{11,t}&\ldots & y_{1M,t} \\
    \vdots &&\vdots \\
    y_{N1,t} &\ldots &y_{NM,t}
\end{pmatrix}\in \mathbb{R}^{N\times M}.
$$ 
Let $\tilde{\mathbf{Y}}_t$ denote the collection of control unit outcomes: $(\vec{y}_{2t},\ldots, \vec{y}_{Nt})^\top$.


In [4]:
import torch
import learnQorthogonal as lqOrth

(CVXPY) Sep 18 03:13:54 PM: Encountered unexpected exception importing solver DIFFCP:
ImportError('diffcp >= 1.0.15 is required')


In [8]:
# number of time points and units
T = 5
N = 10

# number of outcomes and loadings
M = 3
k = 3

#------------------ Ensure stability/existence of solution --------------------#
if (not (N-1 >= M*k)): print("Identifiability condition not met")

# generate a matrix L so that LL^\top has small condition number.
L = torch.randn(M*k, N-1, dtype=torch.float64)
sym = L @ L.T
while torch.linalg.cond(sym) > 100:
    L = torch.randn(M*k, N-1, dtype=torch.float64)
    sym = L @ L.T

print("Condition number of L is ", torch.linalg.cond(L))
print("Condition number of L @ L.T is ", torch.linalg.cond(sym),"\n")
#--------------------------------------------------------------------------------#

# Breaking down the L matrix into blocks representing tilde_Lambda_m for m in 1,...,M.
# print("L matrix, each break is a tilde_Lambda_m:\n")
L_blocks = L.view(M, k, N-1)
# print(L_blocks)
# print("\n")

Lambda_list = []

# Transforming the L blocks into to the actual loadings for each control unit.
for j in range(N-1):
        Lambda = torch.zeros(k, M, dtype=torch.float64)
        for i in range(M):
            Lambda[:, i:i+1] = L_blocks[i][:, j:j+1]
        Lambda_list.append([Lambda])

# The loadings for the treated unit:
Lambda_1 = torch.randn(k, M, dtype=torch.float64)

# start off with a base factor that increases linearly in T
factor_0 = torch.randn(k,1,dtype=torch.float64)
factors = [factor_0 + _*0.1*T for _ in range(T)]

# The control outcomes for time t=1,...,T_0. They are in dimension N-1 times M, ala \tilde{Y}_t
control_outcomes_list = []
for t in range(T):
    # multiply the transposed loadings by the factors at time t.
    control_outcomes_t = [torch.as_tensor(Lambda_list[i][0]).T @ factors[t] for i in range(len(Lambda_list))]
    control_outcomes_t = torch.stack(control_outcomes_t).reshape(N-1,M)
    control_outcomes_t = control_outcomes_t.T
    control_outcomes_list.append(control_outcomes_t)

# These are the treated outcomes over t. They have dimension M times 1. 
# This makes sense: we will subtract \tilde{Y}_t^\top Q w = (M, N-1) (N-1, D)(D,1) = (M,1) from it.
treated_outcomes_list = []
for t in range(T):
    # multiply the transposed loadings by the factors at time t.
    treated_outcomes_t = Lambda_1.T @ factors[t]
    treated_outcomes_list.append(treated_outcomes_t)


Condition number of L is  tensor(9.22, dtype=torch.float64)
Condition number of L @ L.T is  tensor(84.99, dtype=torch.float64) 



Generate data first, then fit with two different seeds, checking that the initial Q matrices are different.

In [13]:
Q_final, w_final = lqOrth.learnQorthogonal(
    targets = treated_outcomes_list,
    covariates = control_outcomes_list,
    embedding_dim = 10,
    n_iterations = 2000,
    reg_Q = 0,
    reg_w = 100,
    verbose = True,
    #init_Q = "eye"
)

print("Target: \n", treated_outcomes_list[T-1])
print("SDM Prediction: \n", control_outcomes_list[T-1] @ Q_final @ w_final)


ortho error: 
 tensor(4.00e-15, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
Step    0 | Loss: 21.06895973
Step    0 | Loss: 21.06895973 | w: [-0.     0.232  0.041  0.23  -0.     0.026  0.346  0.045 -0.     0.079]
Grad norm: 38.72750048
ortho error: 
 tensor(3.61e-15, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
Step  200 | Loss: 9.97216243
Step  200 | Loss: 9.97216243 | w: [-0.    -0.     0.029 -0.     0.416  0.127  0.24   0.188 -0.    -0.   ]
Grad norm: 24.34406318
ortho error: 
 tensor(2.92e-15, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
Step  400 | Loss: 8.65972146
Step  400 | Loss: 8.65972146 | w: [ 0.121  0.433  0.239 -0.    -0.    -0.     0.207 -0.    -0.    -0.   ]
Grad norm: 21.09747014
ortho error: 
 tensor(2.99e-15, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
Step  600 | Loss: 6.99950598
Step  600 | Loss: 6.99950598 | w: [ 0.478  0.171 -0.    -0.    -0.    -0.     0.023  0.073 -0.     0.256]
Grad norm: 17.58006846
ortho 